# 9.2 做市商策略：Avellaneda-Stoikov 模型

## 学习目标
- 理解做市商的盈利来源（价差收入）和风险（库存风险）
- 实现 Avellaneda-Stoikov（2008）做市商最优动态报价模型
- 模拟做市商 PNL 并分析库存风险控制


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.random.seed(42)
print('Libraries loaded')


## 1. 做市商的盈利逻辑

做市商在买卖两侧同时挂单：
- **在 Bid 价格买入**（比市场中间价低 δ）
- **在 Ask 价格卖出**（比市场中间价高 δ）

**理想情况**：双边成交，赚取价差 2δ，库存归零。

**风险**：如果只有一侧成交（如只卖出），库存堆积，承担价格风险。

这就是**库存风险**：持仓越大，价格逆向波动造成的损失越大。


In [ ]:
# 模拟基准价格
np.random.seed(42)
n = 1000
sigma = 0.01  # 价格波动率
S = np.cumsum(np.random.normal(0, sigma, n)) + 100  # 随机游走中间价

# 固定价差报价（基础策略）
delta = 0.05  # 单侧报价偏移
ask = S + delta
bid = S - delta

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(S, 'k-', lw=1, label='中间价')
ax.plot(ask, 'r-', lw=0.8, alpha=0.7, label='做市卖价（Ask）')
ax.plot(bid, 'g-', lw=0.8, alpha=0.7, label='做市买价（Bid）')
ax.fill_between(range(n), bid, ask, alpha=0.1, color='blue', label='价差带')
ax.set_title('做市商价差报价示意图')
ax.set_xlabel('时刻'); ax.set_ylabel('价格')
ax.legend(); plt.tight_layout(); plt.show()


## 2. Avellaneda-Stoikov 模型：最优保留价

**核心思想**：将库存风险纳入定价，使报价随库存动态偏移：

$$r = S - q \cdot \gamma \cdot \sigma^2 \cdot (T - t)$$

$$\delta^{bid} = r - S + \frac{1}{2}\left(\frac{\gamma \sigma^2 (T-t)}{k}\right)$$

- $r$：保留价（做市商的参考中间价，随库存偏移）
- $q$：当前库存（正=过多多头，负=过多空头）
- $\gamma$：风险厌恶系数
- $k$：市场深度参数（控制成交概率）


In [ ]:
# Avellaneda-Stoikov 模型模拟
gamma = 0.1   # 风险厌恶系数
eta   = 0.01  # 市场深度
T     = 1.0   # 总时间

inventory = 0
cash = 0
pnl_list, inv_list, bid_list, ask_list = [], [], [], []

for t in range(n):
    time_left = T - t/n
    # 保留价（随库存偏移使风险降低）
    reservation = S[t] - inventory * gamma * sigma**2 * time_left
    # 最优价差
    optimal_spread = gamma * sigma**2 * time_left + 2/gamma * np.log(1 + gamma/eta)
    b = reservation - optimal_spread/2
    a = reservation + optimal_spread/2
    bid_list.append(b)
    ask_list.append(a)
    # 模拟成交（泊松到达）
    buy_arrival  = np.random.exponential(1/(eta*np.exp(-eta*(S[t]-b)))) < 1.0
    sell_arrival = np.random.exponential(1/(eta*np.exp(-eta*(a-S[t])))) < 1.0
    if buy_arrival:   inventory += 1;  cash -= b
    if sell_arrival:  inventory -= 1;  cash += a
    pnl_list.append(cash + inventory * S[t])
    inv_list.append(inventory)

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
axes[0].plot(S, 'k', lw=0.8, label='中间价')
axes[0].plot(bid_list, 'g', lw=0.8, alpha=0.7, label='AS 买价')
axes[0].plot(ask_list, 'r', lw=0.8, alpha=0.7, label='AS 卖价')
axes[0].set_title('Avellaneda-Stoikov 动态报价')
axes[0].legend(fontsize=8)
axes[1].plot(inv_list, 'blue', lw=1); axes[1].axhline(0, color='gray', lw=0.5)
axes[1].set_title('库存变化（目标：期末归零）')
axes[2].plot(pnl_list, 'green', lw=1.2); axes[2].axhline(0, color='gray', lw=0.5)
axes[2].set_title('累积 PNL')
plt.tight_layout(); plt.show()
print(f'期末库存: {inv_list[-1]}  期末 PNL: {pnl_list[-1]:.2f}')


## 🎯 练习

1. 增大风险厌恶系数 gamma（从 0.1 到 1.0），观察保留价偏移程度和期末库存如何变化。
2. 实现一个简单的库存限制规则：当库存超过 ±10 时，强制单边报价（只挂单让库存回归），计算 PNL。
3. 研究做市商策略与高频交易者（HFT）的关系：HFT 如何通过「抢跑」（front-running）侵蚀做市商利润？

---
**下一节** → `../04_backtesting/10_performance_attribution.ipynb`